# escape.stream_new — StreamSession tutorial

A **StreamSession** is your single handle for live beamline data.  
This notebook runs entirely on **synthetic data** — no beamline connection needed.

Run the cells top-to-bottom; each section covers one concept.

---

| Task | One-liner |
|---|---|
| Connect | `ses = StreamSession('localhost:9999')` |
| What's broadcasting? | `ses.available()` |
| Subscribe (auto-accumulate) | `i0 = ses.i0` |
| Live table | `ses.status()` |
| Spot missing names | `ses.missing()` |
| Per-shot ratio | `ratio = ses.i / ses.i0` |
| Pump-on filter | `i_on = ses.i[ses.pump_on]` |
| Bounded window | `with ses.drift: time.sleep(5)` |
| Blocking burst | `data = gather(ses.i0, ses.t, seconds=5)` |
| Delay-scan plot | `ses.t.digitize(bins).categorize(ses.i / ses.i0).plot_med(update=0.5)` |
| Histogram | `ses.i0.plot_hist(update=0.5)` |
| Correlation scatter | `ses.i.plot_corr(ses.i0, Npoints=300, update=0.5)` |
| Snapshot → DataFrame | `ses.i0.to_frame()` |
| Shut down | `ses.stop()` |

---

### Synthetic stream channels (~50 Hz)

| Channel | Physics |
|---|---|
| `i0` | Incoming flux, Gamma-distributed shot noise |
| `i` | Transmitted signal: pump-probe response × `i0` × slow drift |
| `t` | Pump-probe delay [ps], uniform random on [−1, 10] |
| `pump_on` | Laser state: 1 (on) ≈ 20 % of shots, 0 otherwise |
| `drift` | Slow beam intensity drift (PCHIP random walk) |
| `i_pump` | Raw pump amplitude |

> **Prerequisite:** `%matplotlib widget` (ipympl) for live-updating plots.

In [ ]:
%matplotlib widget

In [ ]:
import sys, time
import numpy as np
import matplotlib.pyplot as plt

from _devpath import prepend_local_checkout_to_path
prepend_local_checkout_to_path()

from escape.stream_new import StreamSession, gather, TestStream

# Synthetic bsread stream on localhost:9999 — ~50 Hz pump-probe data
ts = TestStream(port=9999, interval=0.02)
ts.start()
time.sleep(0.3)   # let the sender bind
print(ts)

---
## 1 — Create a StreamSession

A `StreamSession` wraps one `EventWorker` (one backend connection) and creates
`Stream` objects lazily as you access channel names.

The `source` argument accepts:

| Argument | Meaning |
|---|---|
| `'localhost:9999'` | direct bsread PULL on that host:port |
| `'hostname'` | same, default port 9999 |
| `'bsread'` | PSI bsread dispatcher (production default) |
| `'redis'` | Redis/Dragonfly backend |
| `None` | reuse the existing module-level EventWorker |

By default `auto_accumulate=True` — every channel you access starts collecting data immediately.

In [ ]:
ses = StreamSession('localhost:9999')   # auto_accumulate=True by default
print(ses)

---
## 2 — Discover what channels the backend is broadcasting

`ses.available()` waits briefly for the first event, then reads the channel
names from it.  On the PSI bsread dispatcher it queries the channel search API.

This is how you find out what to subscribe to — the session never guesses or
hard-codes names.

In [ ]:
channels = ses.available()   # waits up to 2 s for first event
print('Available channels:')
for ch in channels:
    print(f'  {ch}')

---
## 3 — Subscribe to channels

Attribute (or item) access creates a `Stream` and calls `accumulate(True)` automatically.
The channel name must match what the backend broadcasts exactly.

```python
i0 = ses.i0        # equivalent to ses['i0']
```

After a short wait you can inspect how many events have been collected.

In [ ]:
i0    = ses.i0        # incoming photon flux
i     = ses.i         # transmitted signal
t     = ses.t         # pump-probe delay [ps]
pump  = ses.pump_on   # laser state (0 / 1)
drift = ses.drift     # slow beam drift

time.sleep(2)   # ~100 events at 50 Hz

print(f'i0    : {len(i0):4d} events')
print(f'i     : {len(i):4d} events')
print(f't     : {len(t):4d} events')
print(f'pump  : {len(pump):4d} events')
print(f'drift : {len(drift):4d} events')

### Status at a glance

`ses.status()` prints a live table of all subscribed channels: event count,
accumulation state (● live / ○ stopped), and a ⚠ flag for channels with no data yet.

In [ ]:
ses.status()

---
## 4 — Spot missing channels

If a channel name doesn't exist on the backend, the `Stream` will have 0 events.
`ses.missing()` returns the names of all zero-event streams — useful for catching
typos or misconfigured channel names before a scan.

In [ ]:
_ = ses['SARBD02-DBPM070:Q1']   # a real SwissFEL name — not in synthetic stream

time.sleep(1)   # wait one second — no events should arrive for this channel

print('Missing channels:', ses.missing())
# → ['SARBD02-DBPM070:Q1']  (0 events received)

---
## 5 — Derived streams: arithmetic and filtering

Operator overloading returns new `Stream` objects backed by a live per-event computation.  
They auto-accumulate via the session and are syntactically identical to `escape.Array`.

| Expression | Result |
|---|---|
| `ses.i / ses.i0` | per-shot normalized signal |
| `ses.i[ses.pump_on]` | values where `pump_on == 1` |
| `ses.i[~ses.pump_on]` | values where `pump_on == 0` (`~` = logical NOT) |
| `ses.i - ses.i0` | difference, per shot |

In [ ]:
ratio = ses.i / ses.i0        # I/I₀ per shot  →  new Stream, auto-accumulates
i_on  = ses.i[ses.pump_on]   # pump-on shots only
i_off = ses.i[~ses.pump_on]  # pump-off shots  (~pump = logical NOT)

time.sleep(3)   # ~150 events; ~30 pump-on, ~120 pump-off (20 % duty cycle)

print(f'ratio : {len(ratio):4d}  (all shots)')
print(f'i_on  : {len(i_on):4d}  (pump on, ≈ 20 % of shots)')
print(f'i_off : {len(i_off):4d}  (pump off)')
print(f'i_on + i_off = {len(i_on) + len(i_off)}  (≈ {len(i)})')
print()

# Pump amplifies the transmitted signal → pump-on mean should be slightly higher
if len(i_on) > 0 and len(i_off) > 0:
    print(f'mean i (pump on)  = {np.array(i_on.data[0]).mean():.4f}')
    print(f'mean i (pump off) = {np.array(i_off.data[0]).mean():.4f}')

---
## 6 — Bounded acquisition window: context manager

`Stream` implements the context manager protocol:

- `__enter__` → `accumulate(True)`
- `__exit__` → `accumulate(False)` — guaranteed even if an exception is raised

You supply the duration with an explicit `time.sleep()`; the block gives you
a precise, self-contained acquisition window for a single channel.

Most useful when `auto_accumulate=False`, or when you want to restart
accumulation on a stopped channel.

In [ ]:
# Stop drift accumulation first so the window starts fresh
ses.drift.accumulate(False)

with ses.drift:
    print('Collecting drift for 5 s…')
    time.sleep(5)
# accumulate(False) was called automatically on exit

n = len(ses.drift)
arr_drift = ses.drift[-n:]
print(f'Collected {n} events in the 5 s window')
print(f'Drift mean = {arr_drift.mean():.4f},  std = {arr_drift.std():.4f}')

### Multiple streams: `contextlib.ExitStack`

For several streams at once, `contextlib.ExitStack` composes any number of context
managers without a helper function:

In [ ]:
from contextlib import ExitStack

# ExitStack applies __enter__/__exit__ to multiple streams at once.
# On exit, accumulate(False) is called on all of them — even if an exception occurs.
with ExitStack() as stk:
    stk.enter_context(ses.i0)
    stk.enter_context(ses.i)
    stk.enter_context(ses.t)
    print('Collecting i0, i, t for 3 s…')
    time.sleep(3)
# All three streams are now stopped.

print(f'i0: {len(ses.i0):4d} events   i: {len(ses.i):4d}   t: {len(ses.t):4d}')

# Restart live accumulation for the rest of the notebook
for s in [ses.i0, ses.i, ses.t]:
    s.accumulate(True)

---
## 7 — `gather()`: blocking multi-stream burst

`gather()` is the preferred idiom for scripted acquisition:

```python
data = gather(*streams, seconds=5)           # fixed duration
data = gather(*streams, n_events=500)        # fixed event count
data = gather(*streams, seconds=30, n_events=500)  # whichever comes first
```

It starts accumulation, blocks, stops, and returns `{name: stream}`.  
Best used with `auto_accumulate=False` so the burst window is clean.

Here we run a **second TestStream on port 9998** for the burst demo, keeping
the live session on 9999 unaffected.

In [ ]:
ts_burst = TestStream(port=9998, interval=0.02)
ts_burst.start()
time.sleep(0.3)

# auto_accumulate=False: nothing starts collecting until explicitly told to
# make_default=False: don't overwrite the live ses's EventWorker as module default
ses_burst = StreamSession('localhost:9998', auto_accumulate=False, make_default=False)
print(ses_burst)

In [ ]:
# Blocking 5-second burst on three channels
data = gather(ses_burst.i0, ses_burst.i, ses_burst.t, seconds=5)

print('Event counts:', {name: len(s) for name, s in data.items()})
print()

# The raw event lists live in stream.data[0]
t_arr  = np.array(data['t'].data[0])
i_arr  = np.array(data['i'].data[0])
i0_arr = np.array(data['i0'].data[0])
ratio  = i_arr / i0_arr

# Bin by delay — same logic as escape.Array.digitize() but with raw numpy
bins = np.linspace(-1, 10, 23)
inds = np.digitize(t_arr, bins)
bc   = 0.5 * (bins[:-1] + bins[1:])
med  = np.array([np.median(ratio[inds == k]) for k in range(1, len(bins))])

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(bc, med, 'o-', ms=4, lw=1.5)
ax.axvline(0, ls='--', c='gray', lw=0.8, label='t = 0')
ax.set_xlabel('delay (ps)')
ax.set_ylabel('I/I₀  (median)')
ax.set_title(f'gather() — 5 s burst,  {len(t_arr)} events')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()

In [ ]:
ses_burst.stop()
ts_burst.stop()

---
## 8 — Single-line analysis chain

The syntax is **identical to `escape.Array`**:

```python
# escape.Array (stored data)
arr_t.digitize(bins).categorize(arr_i / arr_i0).plot_med()

# escape.stream_new (live data) — same call, different objects
ses.t.digitize(bins).categorize(ses.i / ses.i0).plot_med(update=0.5)
```

With `auto_accumulate=True`, every step in the chain auto-subscribes.  
`plot_med()` calls `accumulate(True)` internally, so the chain **fully starts itself**.

### Delay-scan: I/I₀ vs pump-probe delay

In [ ]:
bins = np.linspace(-1, 10, 23)   # 22 delay bins over [-1, 10] ps

fig1, ax1 = plt.subplots(figsize=(9, 3))
ax1.set_title('I/I₀ vs delay — all shots  (live, updating every 0.5 s)')

# Single line — identical syntax to escape.Array
mp1 = ses.t.digitize(bins).categorize(ses.i / ses.i0).plot_med(update=0.5, axes=ax1)

### Pump-on shots only

Filtering and binning can be chained in the same expression:

In [ ]:
fig2, ax2 = plt.subplots(figsize=(9, 3))
ax2.set_title('I/I₀ vs delay — pump-on shots only  (live)')

# (ses.i / ses.i0)[ses.pump_on] = ratio for pump-on events only
ratio_on = (ses.i / ses.i0)[ses.pump_on]
mp2 = ses.t.digitize(bins).categorize(ratio_on).plot_med(update=0.5, axes=ax2)

---
## 9 — Live plots

### Value histogram

In [ ]:
fig_h, ax_h = plt.subplots(figsize=(6, 3))
ax_h.set_title('i0 distribution  (live)')
hp = ses.i0.plot_hist(update=0.5, n_bins=40, axes=ax_h)

### Correlation scatter: I vs I₀

In [ ]:
fig_c, ax_c = plt.subplots(figsize=(4.5, 4.5))
ax_c.set_title('I vs I₀  (last 300 shots)')
cp = ses.i.plot_corr(ses.i0, Npoints=300, update=0.5, axes=ax_c)

---
## 10 — Snapshot as a DataFrame

`stream.to_frame()` exports the accumulated events to a pandas DataFrame.  
In streaming mode (no scan step), columns are `event`, `scan_step`, `scan_value`, and the channel name.
This is useful for feeding data into pandas/seaborn workflows.

In [ ]:
df = ses.i0.to_frame()
print(f'Shape: {df.shape}')
df.head(8)

In [ ]:
# Quick statistics from the snapshot
df.describe()

---
## 11 — Check status after all subscriptions

At this point the session has accumulated several derived streams.  
`ses.status()` gives a full picture before we shut down.

In [ ]:
ses.status()

---
## 12 — Stop

`ses.stop()` calls `accumulate(False)` on all streams and shuts down the
EventWorker's background thread.  Always call this at the end of a session.

In [ ]:
# Stop live-plot update threads
for p in [mp1, mp2, hp, cp]:
    try:
        p.stop()
    except Exception:
        pass

# Stop all streams and the EventWorker
ses.stop()

# Stop the synthetic stream process
ts.stop()

print('Done.')